In [ ]:
import json
import requests
from clickhouse_connect import get_client

#url = "https://raw.githubusercontent.com/ardian28/GeoJson-Indonesia-38-Provinsi/refs/heads/main/Provinsi/38%20Provinsi%20Indonesia%20-%20Provinsi.json"

#data = requests.get(url).json()

client = get_client(host=, port=, username=, password=)

#rows = []
#for f in data['features']:
#    prov = f['properties']['PROVINSI']
#    geom = json.dumps(f['geometry'])
#    rows.append((prov.upper(), geom))  # sekalian UPPER biar aman

#client.insert('geo_provinsi', rows)

/var/folders/49/2b70rr1n58s4z6jst9f201xw0000gn/T/ipykernel_61534/1728690086.py:3: DeprecationWarning: Python 3.9 support is deprecated and will be removed in a future release. This version of clickhouse-connect may stop working with Python 3.9 unexpectedly.
  from clickhouse_connect import get_client


In [ ]:
%pip install clickhouse-connect

Python(53852) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install Elasticsearch

Python(53864) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from elasticsearch import Elasticsearch


/Users/rezkyyayang/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
es = Elasticsearch(
    "http://",
    basic_auth=("", "")
)

In [ ]:
page = es.search(
    index="course",
    body={"size": 1, "query": {"match_all": {}}}
)

one_doc = page["hits"]["hits"][0] if page["hits"]["hits"] else None
one_doc

{'_index': 'course',
 '_id': '9230f36f-72ab-4b25-8fc8-fe4d157b612b',
 '_score': 1.0,
 '_source': {'_class': 'id.go.lan.scheduler.document.CourseDocument',
  'guidUser': '9230f36f-72ab-4b25-8fc8-fe4d157b612b',
  'uuid': 'b4931344-31b2-4eed-bee4-fdf8b4317aaa',
  'nip': '198909192025212046',
  'nama': 'JURNI MAILIA',
  'jabatan': 'Perawat Terampil',
  'instansi': 'Pemerintah Kab. Aceh Besar',
  'pangkat': 'VII',
  'tempatLahir': 'ACEH BESAR',
  'tanggalLahir': '19-09-1989',
  'jenisKelamin': 'F',
  'statusMateri': True,
  'statusJurnal': False,
  'statusJurnalKdua': False,
  'skor': 0.0,
  'sikapPerilaku': 0.0,
  'evaluasiPilihanGanda': 0.0,
  'evaluasiEssay': 0.0,
  'code': 'Gw5KjCoXD0xUHFpDC0jwiLhx4iBDT86YZYk1hVqO',
  'isSelesai': 0,
  'isSelesaiPilihanGanda': 0,
  'isSelesaiEssay': 0,
  'isActive': 1,
  'isRemedial': 0,
  'createAt': '2026-09-25T04:30:51.505Z',
  'updatedAt': '2026-09-25T04:30:51.505Z',
  'createdBy': 'smartbangkom',
  'openedCourseAt': '2026-09-25 00:00:00',
  'closed

In [ ]:
import time

all_docs = []
scroll_size = 10000  # Ambil 1000 dokumen per batch
scroll_timeout = "5m"  # Timeout lebih lama
max_retries = 5  # Maksimal retry per batch

try:
    # Initial search dengan scroll
    page = es.search(
    index="course",
    scroll=scroll_timeout,
    #size=scroll_size,
    body={
        "size": scroll_size,
        "query": {
        "range": {
        "updatedAt": {
        "gte": "2026-10-01T00:00:00.000Z",
        "lt": "2026-11-03T00:00:00.000Z"
        }
        }
        }
    }
    )

    sid = page["_scroll_id"]
    scroll_hits = page["hits"]["hits"]
    all_docs.extend(scroll_hits)

    print(f"Batch 1: {len(scroll_hits)} dokumen")

    # Loop untuk scroll
    batch_num = 2
    consecutive_errors = 0
    max_consecutive_errors = 5  # Stop jika 5 kali error berturut-turut

    while len(scroll_hits) > 0:
        retry_count = 0
        success = False

        while retry_count < max_retries and not success:
            try:
                page = es.scroll(scroll_id=sid, scroll=scroll_timeout)
                sid = page["_scroll_id"]
                scroll_hits = page["hits"]["hits"]

                if len(scroll_hits) > 0:
                    all_docs.extend(scroll_hits)
                    print(f"Batch {batch_num}: {len(scroll_hits)} dokumen (Total: {len(all_docs)})")
                    consecutive_errors = 0  # Reset counter jika berhasil
                    batch_num += 1

                success = True
                # Tambah delay kecil untuk mengurangi beban
                time.sleep(0.2)

            except Exception as e:
                retry_count += 1
                consecutive_errors += 1

                if retry_count < max_retries:
                    wait_time = retry_count * 2  # Exponential backoff
                    print(f"⚠ Error pada batch {batch_num} (attempt {retry_count}/{max_retries}): {type(e).__name__}")
                    print(f"  Retry dalam {wait_time} detik...")
                    time.sleep(wait_time)
                else:
                    print(f"✗ Batch {batch_num} gagal setelah {max_retries} percobaan")
                    print(f"  Error: {e}")

                    # Cek apakah masih bisa lanjut dengan scroll_id baru
                    if consecutive_errors >= max_consecutive_errors:
                        print(f"\n⚠ Terlalu banyak error berturut-turut ({consecutive_errors}x)")
                        print("  Menghentikan proses...")
                        scroll_hits = []  # Stop loop
                        break
                    else:
                        # Coba lanjut ke batch berikutnya (skip yang error)
                        print("  Mencoba lanjut ke batch berikutnya...")
                        break

        if not success and consecutive_errors < max_consecutive_errors:
            # Jika gagal, coba buat scroll baru dari posisi terakhir
            try:
                print(f"  Mencoba scroll baru dari posisi terakhir...")
                page = es.scroll(scroll_id=sid, scroll=scroll_timeout)
                sid = page["_scroll_id"]
                scroll_hits = page["hits"]["hits"]
            except:
                scroll_hits = []  # Stop jika tidak bisa

    print(f"\n✓ Selesai! Total dokumen berhasil diambil: {len(all_docs)}")

except Exception as e:
    print(f"\n✗ Error fatal: {e}")
    print(f"Data yang berhasil diambil: {len(all_docs)} dokumen")

Batch 1: 10000 dokumen
Batch 2: 10000 dokumen (Total: 20000)
Batch 3: 10000 dokumen (Total: 30000)
Batch 4: 10000 dokumen (Total: 40000)
Batch 5: 10000 dokumen (Total: 50000)
Batch 6: 10000 dokumen (Total: 60000)
Batch 7: 10000 dokumen (Total: 70000)
Batch 8: 10000 dokumen (Total: 80000)
Batch 9: 10000 dokumen (Total: 90000)
Batch 10: 10000 dokumen (Total: 100000)
⚠ Error pada batch 11 (attempt 1/5): ConnectionTimeout
  Retry dalam 2 detik...
⚠ Error pada batch 11 (attempt 2/5): ConnectionTimeout
  Retry dalam 4 detik...
Batch 11: 10000 dokumen (Total: 110000)
Batch 12: 10000 dokumen (Total: 120000)
Batch 13: 2633 dokumen (Total: 122633)

✓ Selesai! Total dokumen berhasil diambil: 122633


In [ ]:
# Cek berapa banyak data yang berhasil diambil
print(f"Total dokumen di all_docs: {len(all_docs)}")
print(f"Estimasi batch yang berhasil: {len(all_docs) / 1000:.1f}")

# Cek apakah masih ada data yang tersisa di Elasticsearch
total_count = es.count(index="course", body={"query": {"match_all": {}}})
print(f"\nTotal dokumen di Elasticsearch: {total_count['count']}")
print(f"Data yang terambil: {len(all_docs)} ({len(all_docs)/total_count['count']*100:.1f}%)")

Total dokumen di all_docs: 122633
Estimasi batch yang berhasil: 122.6

Total dokumen di Elasticsearch: 1653669
Data yang terambil: 122633 (7.4%)


In [ ]:
# Konversi ke DataFrame
import pandas as pd

if all_docs:
    df = pd.json_normalize([doc['_source'] for doc in all_docs])
    print(f"\nDataFrame dibuat dengan {len(df)} baris dan {len(df.columns)} kolom")
    print("\nKolom yang tersedia:")
    print(df.columns.tolist())
else:
    print("Tidak ada data untuk dikonversi ke DataFrame")


DataFrame dibuat dengan 122633 baris dan 107 kolom

Kolom yang tersedia:
['_class', 'guidUser', 'uuid', 'nip', 'nama', 'jabatan', 'instansi', 'pangkat', 'tempatLahir', 'tanggalLahir', 'jenisKelamin', 'statusMateri', 'statusJurnal', 'statusJurnalKdua', 'skor', 'sikapPerilaku', 'evaluasiPilihanGanda', 'evaluasiEssay', 'code', 'isSelesai', 'isSelesaiPilihanGanda', 'isSelesaiEssay', 'isActive', 'isRemedial', 'createAt', 'updatedAt', 'createdBy', 'openedCourseAt', 'closedCourseAt', 'openedEvaluationAt', 'ip', 'item', 'openMateri', 'statusProgressMateri', 'statusProgressMateriKDua', 'skorEvaluasiKDua', 'statusEvaluasiKDua', 'totalTrophy', 'masterTypes.id', 'masterTypes.namaType', 'angkatan.id', 'angkatan.nama', 'angkatan.alamat', 'angkatan.daerah', 'angkatan.openEvaluasi', 'angkatan.closeEvaluasi', 'pegawaiDto.nip', 'pegawaiDto.nama', 'pegawaiDto.pnsId', 'pegawaiDto.golRuangAkhir', 'pegawaiDto.jabatanNama', 'pegawaiDto.statusPegawai', 'pegawaiDto.instansiIndukNama', 'pegawaiDto.tempatLahir'

In [ ]:
df.to_csv('course_data_all_lengkap_20261102.csv', index=False)
print(f"DataFrame berhasil disimpan ke 'course_data_all_lengkap_20261102.csv' dengan {len(df)} baris")

DataFrame berhasil disimpan ke 'course_data_all_lengkap_20261102.csv' dengan 122633 baris


In [ ]:
# Set pandas display options to show more rows
import pandas as pd
pd.set_option('display.max_rows', 100)  # Show up to 100 rows
pd.set_option('display.max_columns', None)  # Show all columns

# Or display specific number of rows
df.head(5)  # Show first 50 rows

,_class,guidUser,uuid,nip,nama,jabatan,instansi,pangkat,tempatLahir,tanggalLahir,jenisKelamin,statusMateri,statusJurnal,skor,sikapPerilaku,evaluasiPilihanGanda,evaluasiEssay,code,isSelesai,isSelesaiPilihanGanda,isSelesaiEssay,isActive,isRemedial,createAt,updatedAt,createdBy,openedCourseAt,closedCourseAt,openedEvaluationAt,closedEvaluationAt,ip,item,openMateri,statusProgressMateri,totalTrophy,masterTypes.id,masterTypes.namaType,masterTypes.createdAt,masterTypes.updatedAt,angkatan.id,angkatan.nama,angkatan.alamat,angkatan.daerah,angkatan.openEvaluasi,angkatan.closeEvaluasi,angkatan.createdAt,angkatan.updatedAt,angkatan.isDelete,pegawaiDto.nip,pegawaiDto.nama,pegawaiDto.pnsId,pegawaiDto.golRuangAkhir,pegawaiDto.jabatanNama,pegawaiDto.statusPegawai,pegawaiDto.instansiIndukNama,pegawaiDto.tempatLahir,pegawaiDto.tglLahir,pegawaiDto.jenisKelamin,pegawaiDto.userGuid,pegawaiDto.eselon,pegawaiDto.eselonLevel,pegawaiDto.tmtGolAkhir,pegawaiDto.instansiIndukId,pegawaiDto.instansiKerjaId,pegawaiDto.instansiKerjaKodeCepat,pegawaiDto.instansiKerjaNama,pegawaiDto.jenisJabatan,pegawaiDto.gelarDepan,pegawaiDto.gelarBelakang,pegawaiDto.pendidikanTerakhirNama,pegawaiDto.tkPendidikanTerakhir,pegawaiDto.unorIndukNama,pegawaiDto.unorNama,masterTypes.idSmartbangkom,sertifikatDto.uuid,sertifikatDto.nomorUrutSertifikat,sertifikatDto.nomorSertifikat,sertifikatDto.pathSertifikat,sertifikatDto.penandaTanganPertama,sertifikatDto.penandaTanganKedua,sertifikatDto.guidPenandaTanganPertama,sertifikatDto.guidPenandaTanganKedua,sertifikatDto.dataDiklat.uuid,sertifikatDto.dataDiklat.guidUser,sertifikatDto.dataDiklat.guidAngkatan,sertifikatDto.dataDiklat.createdAt,sertifikatDto.dataDiklat.updatedAt,sertifikatDto.statusTandatangan,sertifikatDto.tahun,sertifikatDto.createdAt,sertifikatDto.updatedAt,sertifikatDto.tglTtePertama,sertifikatDto.tglTteKedua,sertifikatDto.statusGenerateSertifikat
0,id.go.lan.scheduler.document.CourseDocument,f5ed77a6-55e6-430c-8822-11c4621cb1f6,8ee0ce99-83af-47a5-97dc-34bd5dcfaa20,199811082025041006,ARDHIAN ALAMSYAH PRASETYA PUTRA,PENGAWAS TRANSPORTASI DARAT,Pemerintah Provinsi Jawa Timur,III/A,TULUNGAGUNG,08-11-1998,M,True,False,88.0,100.0,88.0,0.0,R3l3qygPECabgJ3kwtX3kQeQwn7Mp7tJ7AS8qEji,0,1,0.0,1,0,2026-02-05T04:28:24.896Z,2026-02-09T07:36:30.933Z,smartbangkom,2026-02-05 00:00:00,2026-02-19 00:00:00,2026-02-19 00:00:00,1770622590926,180.248.34.57,0,1770265704896,True,69.0,ce02a30f-65d9-4b15-8176-11a19fac849d,CPNS,2024-12-16T04:01:32.126Z,2025-09-25T16:06:50.999Z,5bdb0cc4-5357-4dbe-869a-fc7859741f89,latsar 2024,jl.veteran,jakpus,1734282000000,1734282000000,2024-12-16T03:36:22.369Z,2024-12-16T03:36:22.369Z,False,E0mnXHGtSW/WgyPRvMGX9z7YvEARrDOOfBuC5kyrDlE=,awVnLwtUEvj+WcbrRnftRR8IVZKKoHDYiS3PGkIp6oM=,dCop2cCpPR+WtCUxPVhN+YQLzRrE4ozXFiwUKghRvBnwH/...,III/a,PENGAWAS TRANSPORTASI DARAT,CPNS,Pemerintah Provinsi Jawa Timur,TULUNGAGUNG,08-11-1998,M,f5ed77a6-55e6-430c-8822-11c4621cb1f6,Non Eselon,Non Eselon,01-04-2025,A5EB03E23CCCF6A0E040640A040252AD,A5EB03E23CCCF6A0E040640A040252AD,6500,Pemerintah Provinsi Jawa Timur,FUNGSIONAL_UMUM,,,S-1 TEKNIK SIPIL,S-1/Sarjana,UPT Pengelolaan Prasarana Perhubungan Llaj Tul...,Seksi Pengendalian dan Operasional,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,id.go.lan.scheduler.document.CourseDocument,08cd807d-e81f-4493-be9e-91c2bc47a240,6ecba1c8-f499-42da-83d8-f778c7c21ae6,200105052025061009,SAFARI,Pemadam Kebakaran Pemula,Pemerintah Kab. Barito Selatan,II/A,BARITO SELATAN,05-05-2001,M,True,False,86.0,100.0,86.0,0.0,spreI1gKkND4XRRh6mPsHvd5eA4PnXJ25q7B2NV0,0,1,0.0,1,0,2026-02-05T09:33:53.849Z,2026-02-09T07:37:16.707Z,smartbangkom,2026-02-05 00:00:00,2026-02-19 00:00:00,2026-02-19 00:00:00,1770622636693,36.82.252.224,0,1770284033849,True,69.0,ce02a30f-65d9-4b15-8176-11a19fac849d,CPNS,2024-12-16T04:01:32.126Z,2025-09-25T16:06:50.999Z,5bdb0cc4-5357-4dbe-869a-fc7859741f89,latsar 2024,jl.veteran,jakpus,1734282000000,1734282000000,2024-12-16T03:36:22.369Z,2024-12-16T03:36:22.36

In [ ]:
print(df.columns.tolist())

['_class', 'uuid', 'nip', 'nama', 'jabatan', 'instansi', 'pangkat', 'tempatLahir', 'tanggalLahir', 'jenisKelamin', 'statusMateri', 'statusProgressMateri', 'totalTrophy', 'statusJurnal', 'skor', 'sikapPerilaku', 'evaluasiPilihanGanda', 'evaluasiEssay', 'code', 'isSelesai', 'isSelesaiPilihanGanda', 'isSelesaiEssay', 'isActive', 'isRemedial', 'createAt', 'updatedAt', 'createdBy', 'openedCourseAt', 'closedCourseAt', 'openedEvaluationAt', 'closedEvaluationAt', 'ip', 'item', 'openMateri', 'guidUser', 'masterTypes.id', 'masterTypes.namaType', 'masterTypes.idSmartbangkom', 'masterTypes.createdAt', 'masterTypes.updatedAt', 'angkatan.id', 'angkatan.nama', 'angkatan.alamat', 'angkatan.daerah', 'angkatan.openEvaluasi', 'angkatan.closeEvaluasi', 'angkatan.createdAt', 'angkatan.updatedAt', 'angkatan.isDelete', 'pegawaiDto.nip', 'pegawaiDto.nama', 'pegawaiDto.pnsId', 'pegawaiDto.golRuangAkhir', 'pegawaiDto.jabatanNama', 'pegawaiDto.statusPegawai', 'pegawaiDto.instansiIndukNama', 'pegawaiDto.tempatLah

In [ ]:
df_pppk = df[df['masterTypes.namaType'] == 'PPPK']
print(f"Total data PPPK: {len(df_pppk)}")
print(f"Persentase PPPK: {len(df_pppk)/len(df)*100:.2f}%")
df_pppk.head()

Total data PPPK: 3453
Persentase PPPK: 90.46%


,_class,guidUser,uuid,nip,nama,jabatan,instansi,pangkat,tempatLahir,tanggalLahir,...,sertifikatDto.penandaTanganKedua,sertifikatDto.tglTtePertama,sertifikatDto.tglTteKedua,pegawaiDto.tmtGolAkhir,sertifikatDto.dataDiklat.uuid,sertifikatDto.dataDiklat.guidUser,sertifikatDto.dataDiklat.guidAngkatan,sertifikatDto.dataDiklat.createdAt,sertifikatDto.dataDiklat.updatedAt,sertifikatDto.statusGenerateSertifikat
0,id.go.lan.scheduler.document.CourseDocument,da3ed375-b352-4bab-b80d-4de85446f208,2d859258-d2d6-4e9b-b9d8-31892e9cb828,199605012025212101,INTIANA ROSIDAWATI,Bidan Terampil,Pemerintah Kab. Ogan Komering Ulu,,OGAN KOMERING ULU,01-05-1996,...,TRI WIDODO WAHYU UTOMO,2026-04-30T17:00:00.000Z,2026-04-30T17:00:00.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,id.go.lan.scheduler.document.CourseDocument,66844d50-437c-4745-b72e-7a1181f2c12a,e8240086-4b3e-4d8d-856b-d49fd9ed46b1,199004052025211020,WAHYU EKA PUTRA,PENATA LAYANAN OPERASIONAL,Pemerintah Kab. Paser,IX,PASER,05-04-1990,...,TRI WIDODO WAHYU UTOMO,2026-04-30T17:00:00.000Z,2026-04-30T17:00:00.000Z,01-03-2025,eb3b40d9-870b-457a-bfd6-4e0010a2666b,66844d50-437c-4745-b72e-7a1181f2c12a,276b5075-801a-4b2f-9217-323d014f50d7,2026-04-17T01:36:09.318Z,2026-04-17T01:36:09.318Z,NaN
2,id.go.lan.scheduler.document.CourseDocument,4ade4bf2-2a74-44c1-9f4d-52daa9318490,2155acd0-d020-42dc-8856-024234abba79,199112012025212043,SALMIA,Pengelola Sumber Daya Air Ahli Pertama,Kementerian Pekerjaan Umum,IX,SABAH,01-12-1991,...,TRI WIDODO WAHYU UTOMO,2026-04-30T17:00:00.000Z,2026-04-30T17:00:00.000Z,01-10-2025,e0792802-1cef-4c91-a7df-89b2ea919d00,4ade4bf2-2a74-44c1-9f4d-52daa9318490,276b5075-801a-4b2f-9217-323d014f50d7,2026-04-17T05:11:33.194Z,2026-04-17T05:11:33.194Z,NaN
3,id.go.lan.scheduler.document.CourseDocument,93f804e4-6374-4f1e-af33-ad654114b062,ac0647cf-0f25-4de1-9eda-126d611b8d53,199001302025212035,MARINDA PRAJABTI,Pengelola Sumber Daya Air Ahli Pertama,Kementerian Pekerjaan Umum,IX,BULUNGAN,30-01-1990,...,TRI WIDODO WAHYU UTOMO,2026-04-30T17:00:00.000Z,2026-04-30T17:00:00.000Z,01-10-2025,22083a84-7765-4863-a9cd-cdd76572b6c8,93f804e4-6374-4f1e-af33-ad654114b062,276b5075-801a-4b2f-9217-323d014f50d7,2026-04-17T05:11:08.423Z,2026-04-17T05:11:08.423Z,NaN
4,id.go.lan.scheduler.document.CourseDocument,f9ebeb9b-7508-4aa7-a18f-e09826f95589,c7e52125-e490-467c-86ef-8aba0052fa4a,198706182025211046,AEP SAEFULLAH,PENGADMINISTRASI PERKANTORAN,Kementerian Pekerjaan Umum,VII,MAJALENGKA,18-06-1987,...,TRI WIDODO WAHYU UTOMO,2026-04-30T17:00:00.000Z,2026-04-30T17:00:00.000Z,01-10-2025,42717fed-045d-45ad-a355-0dc473e38fe4,f9ebeb9b-7508-4aa7-a18f-e09826f95589,276b5075-801a-4b2f-9217-323d014f50d7,2026-04-17T05:09:43.213Z,2026-04-17T05:09:43.213Z,NaN


In [ ]:
df_pppk.to_csv('course_data_updatedat.csv', index=False)
print(f"DataFrame berhasil disimpan ke 'course_data_updatedat.csv' dengan {len(df_pppk)} baris")

DataFrame berhasil disimpan ke 'course_data_updatedat.csv' dengan 90376 baris


In [ ]:
# Display summary using pandas info and describe methods
print("DataFrame Information:")
print("=" * 60)
df.info()

print("\n\nNumerical Columns Summary:")
print("=" * 60)
print(df.describe())

print("\n\nCategorical Columns Summary:")
print("=" * 60)
for col in df.select_dtypes(include=['object']).columns:
    print(f"\nColumn: {col}")
    print(f"Unique values: {df[col].nunique()}")
    print(f"Top 5 values:\n{df[col].value_counts().head()}")


DataFrame Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35059 entries, 0 to 35058
Columns: 111 entries, _class to masterTypes.idSmartbangkom
dtypes: bool(4), float64(10), int64(9), object(88)
memory usage: 28.8+ MB


Numerical Columns Summary:
               skor  sikapPerilaku  evaluasiPilihanGanda  evaluasiEssay  \
count  35059.000000   35059.000000          35059.000000        35059.0   
mean      58.062320      30.804815             58.062320            0.0   
std       40.591193      45.906210             40.591193            0.0   
min        0.000000       0.000000              0.000000            0.0   
25%        0.000000       0.000000              0.000000            0.0   
50%       82.000000       0.000000             82.000000            0.0   
75%       88.000000     100.000000             88.000000            0.0   
max      100.000000     100.000000            100.000000            0.0   

       isSelesai  isSelesaiPilihanGanda  isSelesaiEssay  isActi

In [ ]:
len(df)

220261

In [ ]:
df.describe()


,totalTrophy,skor,sikapPerilaku,evaluasiPilihanGanda,evaluasiEssay,isSelesai,isSelesaiPilihanGanda,isSelesaiEssay,isActive,isRemedial,item,openMateri,masterTypes.idSmartbangkom,angkatan.openEvaluasi,angkatan.closeEvaluasi,sertifikatDto.statusTandatangan,sertifikatDto.tahun,instansiIndukId,instansiKerjaId
count,219034.000000,220261.000000,220261.000000,220261.000000,220261.0,220261.0,220261.000000,220261.0,220261.0,220261.000000,220261.0,2.202610e+05,0.0,2.202610e+05,2.202610e+05,143076.000000,143076.0,159167.000000,159167.000000
mean,6.975287,76.222954,9.380242,76.222954,0.0,0.0,0.841329,0.0,1.0,0.013765,0.0,1.760943e+12,NaN,1.734466e+12,1.733974e+12,1.313854,2025.0,184.718974,184.876030
std,20.516024,30.024345,29.078645,30.024345,0.0,0.0,0.365370,0.0,0.0,0.116516,0.0,2.659635e+09,NaN,6.430679e+07,1.075975e+08,0.464105,0.0,123.004654,123.304228
min,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,1.0,0.000000,0.0,1.749350e+12,NaN,1.734282e+12,1.733936e+12,0.000000,2025.0,2.000000,2.000000
25%,0.000000,78.000000,0.000000,78.000000,0.0,0.0,1.000000,0.0,1.0,0.000000,0.0,1.759129e+12,NaN,1.734489e+12,1.733936e+12,1.000000,2025.0,104.000000,104.000000
50%,0.000000,86.000000,0.000000,86.000000,0.0,0.0,1.000000,0.0,1.0,0.000000,0.0,1.761186e+12,NaN,1.734489e+12,1.733936e+12,1.000000,2025.0,163.000000,163.000000
75%,0.000000,92.000000,0.000000,92.000000,0.0,0.0,1.000000,0.0,1.0,0.000000,0.0,1.763191e+12,NaN,1.734489e+12,1.733936e+12,2.000000,2025.0,238.000000,238.000000
max,97.000000,100.000000,100.000000,100.000000,0.0,0.0,1.000000,0.0,1.0,1.000000,0.0,1.765439e+12,NaN,1.734489e+12,1.734282e+12,2.000000,2025.0,616.000000,618.000000


In [ ]:
# Load dataset referensi
import pandas as pd

df_reference = pd.read_csv('course_202512150856.csv')
print(f"Dataset referensi loaded: {len(df_reference)} baris")
print(f"Kolom di dataset referensi: {df_reference.columns.tolist()}")

# Ambil daftar NIP dari dataset referensi
nip_reference = set(df_reference['nip'].dropna().astype(str))
print(f"\nTotal NIP unik di dataset referensi: {len(nip_reference)}")

# Filter df: ambil data yang NIPnya TIDAK ADA di dataset referensi
df_filtered = df_pppk[~df_pppk['pegawaiDto.nip'].astype(str).isin(nip_reference)]

print(f"\nHasil filtering:")
print(f"Total data awal: {len(df_pppk)}")
print(f"Data yang NIPnya TIDAK ada di referensi: {len(df_filtered)}")
print(f"Data yang ter-filter: {len(df_pppk) - len(df_filtered)}")

# Simpan hasil ke CSV
df_filtered.to_csv('course_data_filtered_updatedat.csv', index=False)
print(f"\n✓ Data berhasil disimpan ke 'course_data_filtered_updatedat.csv' dengan {len(df_filtered)} baris")

Dataset referensi loaded: 919275 baris
Kolom di dataset referensi: ['id', 'angkatan', 'code', 'created_at', 'evaluasi_akademik', 'ip', 'is_active', 'is_deleted', 'is_selesai', 'item', 'keterangan', 'nip', 'pelatihan', 'tanggal', 'type', 'updated_at']

Total NIP unik di dataset referensi: 919275

Hasil filtering:
Total data awal: 3453
Data yang NIPnya TIDAK ada di referensi: 3453
Data yang ter-filter: 0

✓ Data berhasil disimpan ke 'course_data_filtered_updatedat.csv' dengan 3453 baris


In [ ]:
# Preview data yang ter-filter
print("Sample data yang NIPnya TIDAK ada di dataset referensi:")
df_filtered.head(10)

Sample data yang NIPnya TIDAK ada di dataset referensi:


,_class,uuid,nip,nama,jabatan,instansi,pangkat,tempatLahir,tanggalLahir,jenisKelamin,statusMateri,statusProgressMateri,totalTrophy,statusJurnal,skor,sikapPerilaku,evaluasiPilihanGanda,evaluasiEssay,code,isSelesai,isSelesaiPilihanGanda,isSelesaiEssay,isActive,isRemedial,createAt,updatedAt,createdBy,openedCourseAt,closedCourseAt,openedEvaluationAt,closedEvaluationAt,ip,item,openMateri,guidUser,masterTypes.id,masterTypes.namaType,masterTypes.idSmartbangkom,masterTypes.createdAt,masterTypes.updatedAt,angkatan.id,angkatan.nama,angkatan.alamat,angkatan.daerah,angkatan.openEvaluasi,angkatan.closeEvaluasi,angkatan.createdAt,angkatan.updatedAt,angkatan.isDelete,pegawaiDto.nip,pegawaiDto.nama,pegawaiDto.pnsId,pegawaiDto.golRuangAkhir,pegawaiDto.jabatanNama,pegawaiDto.statusPegawai,pegawaiDto.instansiIndukNama,pegawaiDto.tempatLahir,pegawaiDto.tglLahir,pegawaiDto.jenisKelamin,pegawaiDto.userGuid,pegawaiDto.eselon,pegawaiDto.eselonLevel,pegawaiDto.tmtGolAkhir,pegawaiDto.instansiIndukId,pegawaiDto.instansiKerjaId,pegawaiDto.instansiKerjaKodeCepat,pegawaiDto.instansiKerjaNama,pegawaiDto.jenisJabatan,pegawaiDto.gelarDepan,pegawaiDto.gelarBelakang,pegawaiDto.pendidikanTerakhirNama,pegawaiDto.tkPendidikanTerakhir,pegawaiDto.unorIndukNama,pegawaiDto.unorNama,sertifikatDto.uuid,sertifikatDto.nomorUrutSertifikat,sertifikatDto.nomorSertifikat,sertifikatDto.pathSertifikat,sertifikatDto.penandaTanganPertama,sertifikatDto.penandaTanganKedua,sertifikatDto.guidPenandaTanganPertama,sertifikatDto.guidPenandaTanganKedua,sertifikatDto.dataDiklat.uuid,sertifikatDto.dataDiklat.guidUser,sertifikatDto.dataDiklat.guidAngkatan,sertifikatDto.dataDiklat.createdAt,sertifikatDto.dataDiklat.updatedAt,sertifikatDto.statusTandatangan,sertifikatDto.tahun,sertifikatDto.createdAt,sertifikatDto.updatedAt,sertifikatDto.tglTtePertama,sertifikatDto.tglTteKedua,gelarDepan,golRuangAkhir,tkPendidikanTerakhir,eselon,instansiKerjaNama,statusPegawai,jabatanNama,unorNama,jenisJabatan,instansiKerjaKodeCepat,pendidikanTerakhirNama,gelarBelakang,eselonLevel,instansiIndukId,instansiIndukNama,unorIndukNama,instansiKerjaId,tmtGolAkhir
0,id.go.lan.mooc.document.CourseDocument,ee100497-c810-48a3-960f-00cb443c7317,199905212025051005,MUHAMMAD GHAZI,PENELAAH PENUNTUTAN DAN PENEGAKAN HUKUM,Kejaksaan Agung,III/A,BANDAR LAMPUNG,21-05-1999,M,True,True,69.0,False,76.0,100.0,76.0,0.0,2WopxOmGFNNlH2HD9qSwfc4ia130DDF01aLYOBxE,0,1,0,1,0,2025-09-30T18:26:24.437Z,2025-11-26T14:05:29.818Z,smartbangkom,2025-09-30 00:00:00,2025-10-15 00:00:00,2025-10-15 00:00:00,1759978961601,114.10.101.7,0,1759231584216,09921257-e124-4473-a87c-06c7385790ec,ce02a30f-65d9-4b15-8176-11a19fac849d,CPNS,NaN,2024-12-16T11:01:32.126Z,2025-09-25T23:06:50.999Z,5bdb0cc4-5357-4dbe-869a-fc7859741f89,latsar 2024,jl.veteran,jakpus,1734282000000,1734282000000,2024-12-16T10:36:22.369Z,2024-12-16T10:36:22.369Z,False,199905212025051005,MUHAMMAD GHAZI,iR9O3rEi1F5L07tsAoeQwYGjAvQ59b1Eu9oy2EnLF6KENy...,III/a,PENELAAH PENUNTUTAN DAN PENEGAKAN HUKUM,CPNS,Kejaksaan Agung,BANDAR LAMPUNG,21-05-1999,M,09921257-e124-4473-a87c-06c7385790ec,Non Eselon,Non Eselon,01-05-2025,A5EB03E23BFEF6A0E040640A040252AD,A5EB03E23BFEF6A0E040640A040252AD,4002,Kejaksaan Agung,FUNGSIONAL_UMUM,,S.H.,S-1 ILMU HUKUM,S-1/Sarjana,Kejaksaan Tinggi Lampung,Kejaksaan Negeri Pringsewu,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,id.go.lan.mooc.document.CourseDocument,d5d47b8d-fa91-4fbb-b565-f7358b213eaf,200111132025052007,QATRIN ANGGUN KHAIRUNNISA',Perencana Ahli Pertama,Kejaksaan Agung,III/A,PAMEKASAN,13-11-2001,F,True,True,69.0,False,88.0,100.0,88.0,0.0,sCwyVaRXiMmXxWH5Cj33DGzZsKyM1FSBLo7pAEec,0,1,0,1,0,2025-09-30T21:28:09.659Z,2025-11-26T14:04:16.241Z,smartbangkom,2025-09-30 00:00:00,2025-10-15 00:00:00,2025-10-15 00:00:00,1759985778436,114.8.227.253,0,1759242489619,9a4d4419-e252-40da-bdd8-7724e505d5c2,ce02a30f-65d9-4b15-8176-11a19fac849d,CPNS,NaN,2024-12-16

In [ ]:
# Tampilkan semua kolom di df_filtered
print("Semua kolom di df_filtered:")
print("="*60)
for i, col in enumerate(df_filtered.columns, 1):
    print(f"{col}")

Semua kolom di df_filtered:
_class
uuid
nip
nama
jabatan
instansi
pangkat
tempatLahir
tanggalLahir
jenisKelamin
statusMateri
statusProgressMateri
totalTrophy
statusJurnal
skor
sikapPerilaku
evaluasiPilihanGanda
evaluasiEssay
code
isSelesai
isSelesaiPilihanGanda
isSelesaiEssay
isActive
isRemedial
createAt
updatedAt
createdBy
openedCourseAt
closedCourseAt
openedEvaluationAt
closedEvaluationAt
ip
item
openMateri
guidUser
masterTypes.id
masterTypes.namaType
masterTypes.idSmartbangkom
masterTypes.createdAt
masterTypes.updatedAt
angkatan.id
angkatan.nama
angkatan.alamat
angkatan.daerah
angkatan.openEvaluasi
angkatan.closeEvaluasi
angkatan.createdAt
angkatan.updatedAt
angkatan.isDelete
pegawaiDto.nip
pegawaiDto.nama
pegawaiDto.pnsId
pegawaiDto.golRuangAkhir
pegawaiDto.jabatanNama
pegawaiDto.statusPegawai
pegawaiDto.instansiIndukNama
pegawaiDto.tempatLahir
pegawaiDto.tglLahir
pegawaiDto.jenisKelamin
pegawaiDto.userGuid
pegawaiDto.eselon
pegawaiDto.eselonLevel
pegawaiDto.tmtGolAkhir
pegawaiDto.

In [ ]:
# Daftar kolom yang ingin ditampilkan
selected_columns = [
    'uuid',
    'statusMateri',
    'statusProgressMateri',
    'totalTrophy',
    'statusJurnal',
    'skor',
    'sikapPerilaku',
    'evaluasiPilihanGanda',
    'evaluasiEssay',
    'isSelesai',
    'isSelesaiPilihanGanda',
    'isSelesaiEssay',
    'isActive',
    'isRemedial',
    'createAt',
    'updatedAt',
    'createdBy',
    'openedCourseAt',
    'closedCourseAt',
    'openedEvaluationAt',
    'closedEvaluationAt',
    'ip',
    'masterTypes.namaType',
    'angkatan.nama',
    'pegawaiDto.nip',
    'pegawaiDto.nama',
    'pegawaiDto.pnsId',
    'pegawaiDto.golRuangAkhir',
    'pegawaiDto.jabatanNama',
    'pegawaiDto.statusPegawai',
    'pegawaiDto.instansiIndukNama',
    'pegawaiDto.tempatLahir',
    'pegawaiDto.tglLahir',
    'pegawaiDto.jenisKelamin',
    'pegawaiDto.eselon',
    'pegawaiDto.eselonLevel',
    'pegawaiDto.tmtGolAkhir',
    'pegawaiDto.instansiKerjaNama',
    'pegawaiDto.jenisJabatan',
    'pegawaiDto.gelarDepan',
    'pegawaiDto.gelarBelakang',
    'pegawaiDto.pendidikanTerakhirNama',
    'pegawaiDto.tkPendidikanTerakhir',
    'pegawaiDto.unorIndukNama',
    'pegawaiDto.unorNama',
    'sertifikatDto.nomorUrutSertifikat',
    'sertifikatDto.nomorSertifikat',
    'sertifikatDto.pathSertifikat',
    'sertifikatDto.penandaTanganPertama',
    'sertifikatDto.penandaTanganKedua',
    'sertifikatDto.tglTtePertama',
    'sertifikatDto.tglTteKedua'
]

# Cek kolom mana yang ada di df_filtered
available_columns = [col for col in selected_columns if col in df_filtered.columns]
missing_columns = [col for col in selected_columns if col not in df_filtered.columns]

print(f"Kolom yang tersedia: {len(available_columns)}/{len(selected_columns)}")
if missing_columns:
    print(f"\nKolom yang tidak ditemukan ({len(missing_columns)}):")
    for col in missing_columns:
        print(f"  - {col}")

# Buat dataframe baru dengan kolom yang dipilih
df_selected = df_filtered[available_columns].copy()

print(f"\n✓ DataFrame baru dibuat dengan {len(df_selected)} baris dan {len(df_selected.columns)} kolom")
print(f"\nPreview data:")
df_selected.head()

Kolom yang tersedia: 52/52

✓ DataFrame baru dibuat dengan 3453 baris dan 52 kolom

Preview data:


,uuid,statusMateri,statusProgressMateri,totalTrophy,statusJurnal,skor,sikapPerilaku,evaluasiPilihanGanda,evaluasiEssay,isSelesai,...,pegawaiDto.tkPendidikanTerakhir,pegawaiDto.unorIndukNama,pegawaiDto.unorNama,sertifikatDto.nomorUrutSertifikat,sertifikatDto.nomorSertifikat,sertifikatDto.pathSertifikat,sertifikatDto.penandaTanganPertama,sertifikatDto.penandaTanganKedua,sertifikatDto.tglTtePertama,sertifikatDto.tglTteKedua
0,2d859258-d2d6-4e9b-b9d8-31892e9cb828,True,False,0,True,94.0,0.0,94.0,0.0,0,...,Diploma III/Sarjana Muda,UPTD DINAS KESEHATAN,UPTD PUSKESMAS BATUMARTA II,113937,00113937/PPPK/4012/246/LAN-LAN/2026,2026/04/2d859258-d2d6-4e9b-b9d8-31892e9cb828.pdf,DENY JUNANTO,TRI WIDODO WAHYU UTOMO,2026-04-30T17:00:00.000Z,2026-04-30T17:00:00.000Z
1,e8240086-4b3e-4d8d-856b-d49fd9ed46b1,True,True,0,True,100.0,0.0,100.0,0.0,0,...,S-1/Sarjana,"DINAS KOMUNIKASI, INFORMATIKA, STATISTIK DAN P...",BIDANG TEKNOLOGI KOMUNIKASI DAN INFORMATIKA,113935,00113935/PPPK/4012/246/LAN-LAN/2026,2026/04/e8240086-4b3e-4d8d-856b-d49fd9ed46b1.pdf,DENY JUNANTO,TRI WIDODO WAHYU UTOMO,2026-04-30T17:00:00.000Z,2026-04-30T17:00:00.000Z
2,2155acd0-d020-42dc-8856-024234abba79,True,True,0,True,94.0,0.0,94.0,0.0,0,...,S-1/Sarjana,Kementerian Pekerjaan Umum,Direktorat Jenderal Sumber Daya Air,113933,00113933/PPPK/4012/246/LAN-LAN/2026,2026/04/2155acd0-d020-42dc-8856-024234abba79.pdf,DENY JUNANTO,TRI WIDODO WAHYU UTOMO,2026-04-30T17:00:00.000Z,2026-04-30T17:00:00.000Z
3,ac0647cf-0f25-4de1-9eda-126d611b8d53,True,True,0,True,96.0,0.0,96.0,0.0,0,...,S-1/Sarjana,Kementerian Pekerjaan Umum,Direktorat Jenderal Sumber Daya Air,113932,00113932/PPPK/4012/246/LAN-LAN/2026,2026/04/ac0647cf-0f25-4de1-9eda-126d611b8d53.pdf,DENY JUNANTO,TRI WIDODO WAHYU UTOMO,2026-04-30T17:00:00.000Z,2026-04-30T17:00:00.000Z
4,c7e52125-e490-467c-86ef-8aba0052fa4a,True,True,0,True,98.0,0.0,98.0,0.0,0,...,Diploma III/Sarjana Muda,Kementerian Pekerjaan Umum,Direktorat Jenderal Sumber Daya Air,113928,00113928/PPPK/4012/246/LAN-LAN/2026,2026/04/c7e52125-e490-467c-86ef-8aba0052fa4a.pdf,DENY JUNANTO,TRI WIDODO WAHYU UTOMO,2026-04-30T17:00:00.000Z,2026-04-30T17:00:00.000Z


In [ ]:
# Simpan df_selected ke CSV
df_selected.to_csv('course_data_filtered_selected_updatedat.csv', index=False)
print(f"✓ DataFrame berhasil disimpan ke 'course_data_filtered_selected_updatedat.csv'")
print(f"  Total baris: {len(df_selected)}")
print(f"  Total kolom: {len(df_selected.columns)}")

✓ DataFrame berhasil disimpan ke 'course_data_filtered_selected_updatedat.csv'
  Total baris: 3453
  Total kolom: 52


In [ ]:
# Cari value 'ag93cZ9nM2Cxc/dko0io3E+pOghLMg6qBm2gKGZT+x0=' di semua kolom
search_value = 'ag93cZ9nM2Cxc/dko0io3E+pOghLMg6qBm2gKGZT+x0='

print(f"Mencari value: {search_value}")
print("="*60)

found_in = []

# Cari di semua kolom df
for col in df.columns:
    try:
        # Konversi kolom ke string untuk pencarian
        if df[col].astype(str).str.contains(search_value, na=False, regex=False).any():
            count = df[col].astype(str).str.contains(search_value, na=False, regex=False).sum()
            found_in.append((col, count))
            print(f"✓ Ditemukan di kolom: {col}")
            print(f"  Total kemunculan: {count}")

            # Tampilkan beberapa contoh baris yang mengandung value ini
            matching_rows = df[df[col].astype(str).str.contains(search_value, na=False, regex=False)]
            print(f"  Sample data (3 baris pertama):")
            print(f"  {matching_rows[[col]].head(3).to_string()}")
            print()
    except:
        pass

if not found_in:
    print("✗ Value tidak ditemukan di semua kolom")
else:
    print(f"\nRingkasan: Value ditemukan di {len(found_in)} kolom")

Mencari value: ag93cZ9nM2Cxc/dko0io3E+pOghLMg6qBm2gKGZT+x0=
✗ Value tidak ditemukan di semua kolom


In [ ]:
# Cek value kolom pegawaiDto.nip untuk baris dengan index 3272
try:
    nip_value = df.loc[3272, 'pegawaiDto.nip']
    print(f"Value kolom 'pegawaiDto.nip' pada baris index 3272:")
    print(f"{nip_value}")
    print(f"\nTipe data: {type(nip_value)}")

    # Tampilkan beberapa kolom lain dari baris yang sama untuk konteks
    print(f"\nData lengkap baris 3272:")
    print(df.loc[3272, ['uuid','nip', 'pegawaiDto.nip', 'pegawaiDto.nama', 'pegawaiDto.statusPegawai']].to_string())
except KeyError as e:
    print(f"Error: {e}")
    print(f"Index 3272 tidak ditemukan atau kolom tidak ada")
    print(f"Total baris di df: {len(df)}")
    print(f"Index range: {df.index.min()} - {df.index.max()}")

Value kolom 'pegawaiDto.nip' pada baris index 3272:
NLpBKaW44naHlA6Pur7yLVuRU2ZyDeyqjuumHsuI9ow=

Tipe data: <class 'str'>

Data lengkap baris 3272:
uuid                                a9d85386-6ed2-4356-979a-8cdb2250306d
nip                                                   200002122025061009
pegawaiDto.nip              NLpBKaW44naHlA6Pur7yLVuRU2ZyDeyqjuumHsuI9ow=
pegawaiDto.nama                                 dM0QBHEOkWpwHZSA4PSzbA==
pegawaiDto.statusPegawai                                            CPNS


In [ ]:
print("Ping:", es.ping())
print("Info:", es.info())

Ping: False
Info: {'name': 'elastic-master-1', 'cluster_name': 'elastic-sibangkom', 'cluster_uuid': '_2tiJKM_Sk681A-HxXeJtg', 'version': {'number': '9.2.4', 'build_flavor': 'default', 'build_type': 'deb', 'build_hash': 'dfc5c38614c29a598e132c035b66160d3d350894', 'build_date': '2026-01-08T22:07:25.170027027Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [ ]:
%pip install instaloader
%pip install TikTokApi
%pip install playwright
%playwright install

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 65 kB 2.3 MB/s eta 0:00:01
     |████████████████████████████████| 73 kB 5.4 MB/s eta 0:00:01
     |████████████████████████████████| 41.0 MB 890 kB/s eta 0:00:01
     |████████████████████████████████| 113 kB 3.3 MB/s eta 0:00:01
     |████████████████████████████████| 78 kB 12.2 MB/s eta 0:00:01
     |████████████████████████████████| 274 kB 2.1 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because norma

UsageError: Line magic function `%playwright` not found.


In [ ]:
import instaloader

url = "https://www.instagram.com/p/DVK2QbIEmPl/"

# ambil shortcode dari url
shortcode = url.split("/p/")[1].split("/")[0]

L = instaloader.Instaloader()

post = instaloader.Post.from_shortcode(L.context, shortcode)

print("Likes:", post.likes)
print("Comments:", post.comments)

JSON Query to graphql/query: 403 Forbidden when accessing https://www.instagram.com/graphql/query [retrying; skip with ^C]


Likes: 97
Comments: 2


In [ ]:
import instaloader

url = "https://www.instagram.com/reel/DS8t0GckoS0/"

shortcode = url.rstrip("/").split("/")[-1]

L = instaloader.Instaloader()

post = instaloader.Post.from_shortcode(L.context, shortcode)

print("Username:", post.owner_username)
print("Likes:", post.likes)
print("Comments:", post.comments)
print("Views:", post.video_view_count)
print("Caption:", post.caption[:100])
print("Date:", post.date)

JSON Query to graphql/query: 403 Forbidden when accessing https://www.instagram.com/graphql/query [retrying; skip with ^C]


Username: perupadata
Likes: 98
Comments: 2
Views: None
Caption: Hasil survei Kemenhub, 143,9 juta orang menyatakan akan mudik lebaran. Tapi lewat rilis yang sama, d
Date: 2026-02-25 05:40:31


In [ ]:
import instaloader

url = "https://www.instagram.com/perupadata/"

username = url.rstrip("/").split("/")[-1]

L = instaloader.Instaloader()

profile = instaloader.Profile.from_username(L.context, username)

print("Username:", profile.username)
print("Followers:", profile.followers)
print("Following:", profile.followees)
print("Total Posts:", profile.mediacount)
print("Total Videos:", profile.igtvcount)
print("Bio:", profile.biography)
print("Verified:", profile.is_verified)

Username: perupadata
Followers: 146260
Following: 74
Total Posts: 4179
Total Videos: 1
Bio: Data aktual, tersaji visual
Verified: True


In [ ]:
%pip install yt_dlp

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 3.2 MB 3.2 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import yt_dlp

url = "https://www.tiktok.com/@gadismatirasa889/video/7577435015525551380?is_from_webapp=1&sender_device=pc"

ydl = yt_dlp.YoutubeDL()

info = ydl.extract_info(url, download=False)

print("Views:", info["view_count"])
print("Likes:", info["like_count"])
print("Comments:", info["comment_count"])
print("Uploader:", info["uploader"])
print("Duration:", info["duration"])

Deprecated Feature: Support for Python version 3.9 has been deprecated. Please update to Python 3.10 or above


[TikTok] Extracting URL: https://www.tiktok.com/@gadismatirasa889/video/7577435015525551380?is_from_webapp=1&sender_device=pc
[TikTok] 7577435015525551380: Downloading webpage


/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/sre_parse.py:499: RuntimeWarning: coroutine 'get_tiktok_video' was never awaited
  sourceget = source.get


Views: 34600000
Likes: 2100000
Comments: 3964
Uploader: gadismatirasa889
Duration: 13


In [ ]:
%pip install browser-cookie3

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 207 kB 1.1 MB/s eta 0:00:01
     |████████████████████████████████| 2.5 MB 591 kB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import yt_dlp
from datetime import datetime


url = "https://www.tiktok.com/@gadismatirasa889/video/7577435015525551380"

ydl_opts = {
    'quiet': False,
    'cookiefile': None,
    'cookiesfrombrowser': ('edge',),
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=False)

# convert tanggal
upload_date = info.get("upload_date")

if upload_date:
    upload_date = datetime.strptime(upload_date, "%Y%m%d")

print("Username:", info.get("uploader"))
print("Date:", upload_date)
print("Caption:", info.get("description"))
print("Views:", info.get("view_count"))
print("Likes:", info.get("like_count"))
print("Comments:", info.get("comment_count"))
print("Duration:", info.get("duration"))

Deprecated Feature: Support for Python version 3.9 has been deprecated. Please update to Python 3.10 or above


[TikTok] Extracting URL: https://www.tiktok.com/@gadismatirasa889/video/7577435015525551380
[TikTok] 7577435015525551380: Downloading webpage
Extracting cookies from edge
Extracted 2301 cookies from edge


ERROR: [TikTok] 7577435015525551380: Unable to extract webpage video data; please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U


DownloadError: ERROR: [TikTok] 7577435015525551380: Unable to extract webpage video data; please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U

In [ ]:
%pip install TikTokApi

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from TikTokApi import TikTokApi

api = TikTokApi()

video = api.video(url="https://www.tiktok.com/@gadismatirasa889/video/7577435015525551380")

data = video.info()

print(data["desc"])
print(data["stats"]["playCount"])
print(data["stats"]["diggCount"])
print(data["stats"]["commentCount"])
print(data["createTime"])

Exception: No sessions created, please create sessions first